# Momentum & Adam on 2D Contours

**Momentum** adds velocity; **Adam** adapts per-parameter step sizes.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. Rosenbrock-like elongated bowl

In [ ]:
def loss2d(w):
    return (1 - w[0])**2 + 10 * (w[1] - w[0]**2)**2

w1, w2 = np.meshgrid(np.linspace(-2, 2, 100), np.linspace(-1, 3, 100))
Z = (1 - w1)**2 + 10 * (w2 - w1**2)**2

## 2. Contour plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.contour(w1, w2, Z, levels=30, cmap='viridis')
ax.set_xlabel('w1'); ax.set_ylabel('w2')
ax.set_title('Non-convex optimization landscape')
plt.tight_layout(); plt.show()

## 3. Run SGD, SGD+momentum, Adam

In [ ]:
def track(optimizer_cls, steps=60, lr=0.01, **kw):
    w = torch.tensor([-1.5, 2.0], requires_grad=True)
    opt = optimizer_cls([w], lr=lr, **kw)
    path = [w.detach().clone().numpy()]
    for _ in range(steps):
        l = loss2d(w)
        opt.zero_grad(); l.backward(); opt.step()
        path.append(w.detach().clone().numpy())
    return np.array(path)

p_sgd = track(torch.optim.SGD, lr=0.002)
p_mom = track(torch.optim.SGD, lr=0.002, momentum=0.9)
p_adam = track(torch.optim.Adam, lr=0.05)

## 4. Overlay paths on contours

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.contour(w1, w2, Z, levels=30, cmap='viridis', alpha=0.7)
ax.plot(p_sgd[:,0], p_sgd[:,1], 'r.-', label='SGD', lw=1.5)
ax.plot(p_mom[:,0], p_mom[:,1], 'b.-', label='SGD+momentum', lw=1.5)
ax.plot(p_adam[:,0], p_adam[:,1], 'g.-', label='Adam', lw=1.5)
ax.legend(); ax.set_title('Optimizer paths compared')
plt.tight_layout(); plt.show()

## 5. Loss along paths

In [ ]:
def path_loss(p):
    return [loss2d(torch.tensor(pt)).item() for pt in p]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(path_loss(p_sgd), 'r-', label='SGD')
ax.plot(path_loss(p_mom), 'b-', label='momentum')
ax.plot(path_loss(p_adam), 'g-', label='Adam')
ax.set_xlabel('step'); ax.set_ylabel('loss'); ax.legend()
plt.tight_layout(); plt.show()